# FetchPush closest-regime reproduction

**Purpose (bounded test, NOT tuning):** the vanilla FetchPush binary-NCE baseline
fails; controlled ablations showed it is *not* the objective/relabeling (identical
to the original) and *not* entropy. The remaining main discrepancy vs the original
repo is **training-protocol scale** (1M steps, ~64 SGD updates/env-step, 4-10 actors).
This notebook aligns our regime toward the original as far as a single Colab GPU
allows, and asks one question: **does protocol scale close the gap?**

Fixed (NOT changed): env reset = start-at-object, binary NCE, `use_td=False`,
`twin_q=False`, `random_goals=0.5`, `entropy_coefficient=None` + `target_entropy=0.0`
(matches the original image config), `batch_size=256`. **No causal changes.**

**Gate:** run to 100k; if success is flat and the policy still freezes/disengages
the object, STOP. Only continue to 200k if there is a real improving trend.

In [ ]:
# 1. Install + GPU.
!pip -q install "jax[cuda12]" dm-haiku optax gymnasium gymnasium-robotics mujoco imageio
import os
os.environ["MUJOCO_GL"] = "egl"
import jax; print("JAX devices:", jax.devices())

In [ ]:
# 2. Get / update code.
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
!git pull

In [ ]:
# 3. Mount Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. ===== knobs =====
SGD_PER_ENV_STEP = 16   # 16 / 32 / 64  -- gradient updates PER ENV STEP (original repo used 64)
STEPS = 100_000         # early gate; continue to 200_000 only if improving (cell 10)
SEED = 0
# In our single-process loop, updates_per_step sets the update RATIO and
# num_sgd_steps_per_step batches them on the GPU -- we set both = SGD_PER_ENV_STEP.
RUN_DIR = f"/content/drive/MyDrive/contrastive_rl_runs/fpush_sao_regime_s{SEED}_sgd{SGD_PER_ENV_STEP}"
REPORT_DIR = f"/content/drive/MyDrive/repro_report_fpush_sao_regime_sgd{SGD_PER_ENV_STEP}"
print("updates/env-step:", SGD_PER_ENV_STEP, "| steps:", STEPS, "| RUN_DIR:", RUN_DIR)
print("Est. Colab T4 time to 100k:  sgd16 ~30-50 min | sgd32 ~50-80 min | sgd64 ~90-120 min")

In [ ]:
# 5. Regime comparison: original repo  vs  our aligned Colab regime  vs  remaining unmatched.
rows = [
  ("objective",        "binary NCE, random_goals=0.5", "binary NCE, random_goals=0.5", "= matched"),
  ("td / twin_q",      "False / False",                "False / False",                "= matched"),
  ("entropy",          "image: None(adaptive,t=0)",    "None(adaptive, target=0.0)",   "= matched"),
  ("reset",            "image: start-at-object",       "start-at-object",              "= matched"),
  ("relabeling",       "geometric gamma^k, no filter", "geometric gamma^k, no filter", "= matched"),
  ("batch_size",       "256",                          "256",                          "= matched"),
  ("updates/env-step", "~64 (rate-limited)",           f"{SGD_PER_ENV_STEP}",          "~ partial"),
  ("total steps",      "1,000,000",                    f"{STEPS:,} (->200k)",          "UNMATCHED (5-10x)"),
  ("actors / data",    "4-10 parallel",                "1 (single process)",           "UNMATCHED (diversity)"),
  ("object start",     "image: FIXED [1.15,0.75]",     "randomized",                   "UNMATCHED"),
  ("observation",      "image: 64x64 (or state)",      "state (25)",                   "UNMATCHED if paper=image"),
  ("base env",         "gym robotics v1",              "gymnasium FetchPush-v4",       "minor"),
]
print(f"{'aspect':16} | {'original repo':30} | {'our aligned Colab':28} | status")
print('-'*100)
for a,o,c,s in rows:
    print(f"{a:16} | {o:30} | {c:28} | {s}")

In [ ]:
# 6. Train to the 100k gate.
from crl.config import Config
from crl.train import train
config = Config(
    env_name="fetch_push_start_at_obj",
    use_td=False, twin_q=False, random_goals=0.5,
    entropy_coefficient=None, target_entropy=0.0,   # adaptive entropy (matches original image)
    batch_size=256,
    max_number_of_steps=STEPS,
    min_replay_size=10_000, random_steps=10_000,
    updates_per_step=SGD_PER_ENV_STEP,              # <-- updates PER ENV STEP
    num_sgd_steps_per_step=SGD_PER_ENV_STEP,        # <-- GPU batching of those updates
    eval_every_steps=10_000, eval_episodes=20,
    seed=SEED, ckpt_dir=RUN_DIR, resume=False,
)
assert config.use_td is False and config.twin_q is False
assert config.entropy_coefficient is None and config.random_goals == 0.5
print("OK: fetch_push_start_at_obj, binary NCE, adaptive entropy,",
      f"{SGD_PER_ENV_STEP} updates/env-step -> {RUN_DIR}")
state = train(config)

In [ ]:
# 7. Failure diagnosis at the gate: object displacement, positive-goal spread,
#    contact, action saturation, disengagement (trained vs random).
!python -m crl.diagnose_push --env_name fetch_push_start_at_obj --ckpt {RUN_DIR}/best.pkl --episodes 50

In [ ]:
# 8. Report: success / final_dist / min_dist / logits_gap / categorical_accuracy + GIFs.
!python -m crl.repro_report --env_name fetch_push_start_at_obj --batch_size 256 --run_dirs {RUN_DIR} --out {REPORT_DIR}
import os
from IPython.display import Image, display
for f in ["learning_curves.png", "distance_curves.png", "contrastive_logits.png",
          "ranking_accuracy.png", "rollout_random.gif", "rollout_trained.gif"]:
    p = os.path.join(REPORT_DIR, f)
    if os.path.exists(p):
        print(f); display(Image(p))

In [ ]:
# 9. Montage of the trained rollout (does the gripper push the object, or disengage?).
from crl.visualize import rollout_gif
import imageio.v3 as iio, numpy as np, matplotlib.pyplot as plt
g = rollout_gif("fetch_push_start_at_obj", ckpt=f"{RUN_DIR}/best.pkl",
                out=f"{REPORT_DIR}/rollout_trained.gif", episodes=3)
frames = iio.imread(g); idx = np.linspace(0, 49, 6).astype(int)  # first episode
fig, ax = plt.subplots(1, 6, figsize=(18, 3.2))
for a, i in zip(ax, idx):
    a.imshow(frames[i]); a.axis('off'); a.set_title(f"t={i}")
fig.suptitle("trained rollout ep1: gripper starts ON object -- does it push toward goal?")
plt.show()

## Decision at the 100k gate
- **Flat** (success ~0.05-0.10, `final_dist` not decreasing, trained object displacement <= random, positive-spread still ~0, gripper disengages) -> **STOP**. Conclusion: protocol scale (at single-actor / <=100k) does not close the gap; the remaining unmatched differences are total steps (1M), multi-actor data diversity, and the image env's fixed object.
- **Improving** (success rising, `final_dist` decreasing, displacement > random, positive-spread up) -> run cell 10 to continue to 200k.

In [ ]:
# 10. (only if improving) continue the SAME run to 200k.
config.max_number_of_steps = 200_000
config.resume = True
state = train(config)
# then re-run cells 7-9 to re-diagnose at 200k.